# Imports

In [1]:
from copy import deepcopy
import heapq

# How To Store The States?

## Approach 1: Matrix (3x3) representation

In [2]:
def puzzle_matrix(tile_to_move=None, state=None):

    # 1) Create default state if not provided
    if state is None:
        state = [[1, 2, 3],
                 [4, 0, 5],
                 [6, 7, 8]]

    #  print matrix
    def pm(mat):
        for row in mat:
            print(row)
        print()

    # find tile position
    def find_pos(st, val):
        for r in range(3):
            for c in range(3):
                if st[r][c] == val:
                    return r, c
        return None

    # Print initial state
    print("Initial state:")
    pm(state)

    # If no movement → return initial state
    if tile_to_move is None:
        return state

    # Copy state
    new_state = deepcopy(state)

    # Find positions
    zr, zc = find_pos(new_state, 0)
    tr, tc = find_pos(new_state, tile_to_move)

    # Check adjacency
    if abs(zr - tr) + abs(zc - tc) != 1:
        raise ValueError("selected tile is not adjacent to 0")

    # Swap
    new_state[zr][zc], new_state[tr][tc] = new_state[tr][tc], new_state[zr][zc]

    # Print final state
    print(f"After moving tile {tile_to_move}:")
    pm(new_state)

    return new_state


In [3]:
puzzle_matrix(tile_to_move= 5, state=None)


Initial state:
[1, 2, 3]
[4, 0, 5]
[6, 7, 8]

After moving tile 5:
[1, 2, 3]
[4, 5, 0]
[6, 7, 8]



[[1, 2, 3], [4, 5, 0], [6, 7, 8]]

## Aproach 2: List Representation

In [4]:
def puzzle_list(tile_to_move=None, state=None):
    

    # 1) Default state
    if state is None:
        state = [1, 2, 3,
                 4, 0, 5,
                 6, 7, 8]

    def print_list(lst):
        print(lst[0:3])
        print(lst[3:6])
        print(lst[6:9])
        print()

    # find tile index
    def find_index(lst, val):
        return lst.index(val)

    print("Initial state:")
    print_list(state)

    if tile_to_move is None:
        return state

    new_state = deepcopy(state)

    # positions
    zi = find_index(new_state, 0)
    ti = find_index(new_state, tile_to_move)

    # compute row/col
    zr, zc = divmod(zi, 3)
    tr, tc = divmod(ti, 3)

    # adjacency check
    if abs(zr - tr) + abs(zc - tc) != 1:
        raise ValueError("selected tile is not adjacent to 0")

    # swap
    new_state[zi], new_state[ti] = new_state[ti], new_state[zi]

    print(f"After moving tile {tile_to_move}:")
    print_list(new_state)

    return new_state


In [5]:
puzzle_list(tile_to_move=4, state=None)


Initial state:
[1, 2, 3]
[4, 0, 5]
[6, 7, 8]

After moving tile 4:
[1, 2, 3]
[0, 4, 5]
[6, 7, 8]



[1, 2, 3, 0, 4, 5, 6, 7, 8]

## Aproach 3: String Representation

In [6]:
def puzzle_string(tile_to_move=None, state=None):
    # default
    if state is None:
        state = "123405678"  

    def print_string(s):
        print(s[0:3])
        print(s[3:6])
        print(s[6:9])
        print()

    print("Initial state:")
    print_string(state)

    if tile_to_move is None:
        return state

    # find indexes
    zi = state.index('0')
    ti = state.index(str(tile_to_move))

    zr, zc = divmod(zi, 3)
    tr, tc = divmod(ti, 3)

    if abs(zr - tr) + abs(zc - tc) != 1:
        raise ValueError("selected tile is not adjacent to 0")

    # swap in string
    lst = list(state)
    lst[zi], lst[ti] = lst[ti], lst[zi]
    new_state = "".join(lst)

    print(f"After moving tile {tile_to_move}:")
    print_string(new_state)

    return new_state


In [7]:
puzzle_string(tile_to_move= 2 , state=None)


Initial state:
123
405
678

After moving tile 2:
103
425
678



'103425678'

## Aproach 4: Number Representation

In [8]:
def puzzle_number(tile_to_move=None, state=None):
    # default
    if state is None:
        state = 123405678   

    # turn into a 9 digit string
    s = str(state).zfill(9)

    def print_string(s):
        print(s[0:3])
        print(s[3:6])
        print(s[6:9])
        print()

    print("Initial state:")
    print_string(s)

    if tile_to_move is None:
        return state

    zi = s.index('0')
    ti = s.index(str(tile_to_move))

    zr, zc = divmod(zi, 3)
    tr, tc = divmod(ti, 3)

    if abs(zr - tr) + abs(zc - tc) != 1:
        raise ValueError("selected tile is not adjacent to 0")

    lst = list(s)
    lst[zi], lst[ti] = lst[ti], lst[zi]
    new_s = "".join(lst)

    print(f"After moving tile {tile_to_move}:")
    print_string(new_s)

    return int(new_s)


In [9]:
puzzle_number(tile_to_move=7 , state=None)

Initial state:
123
405
678

After moving tile 7:
123
475
608



123475608

# Required Functions for A* 


## 1: A Function For Generating Children

In [10]:
def generate_children(state):
    """
    Input:  state = 3x3 list
    Output: list of new states (children)
    """

    # find blank (0)
    zr = zc = None
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                zr, zc = r, c
                break
        if zr is not None:
            break

    # possible moves: (dr, dc)
    moves = [
        (-1, 0),  # up
        (1, 0),   # down
        (0, -1),  # left
        (0, 1)    # right
    ]

    children = []

    for dr, dc in moves:
        nr, nc = zr + dr, zc + dc
        # check bounds
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = deepcopy(state)
            # swap blank with neighbor
            new_state[zr][zc], new_state[nr][nc] = new_state[nr][nc], new_state[zr][zc]
            children.append(new_state)

    return children


In [11]:
state = [[1, 2, 3],
         [4, 0, 5],
         [6, 7, 8]]
generate_children(state)

[[[1, 0, 3], [4, 2, 5], [6, 7, 8]],
 [[1, 2, 3], [4, 7, 5], [6, 0, 8]],
 [[1, 2, 3], [0, 4, 5], [6, 7, 8]],
 [[1, 2, 3], [4, 5, 0], [6, 7, 8]]]

In [12]:
state = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
generate_children(state)

[[[3, 1, 2], [0, 4, 5], [6, 7, 8]], [[1, 0, 2], [3, 4, 5], [6, 7, 8]]]

## 2: Heuristic Functions

### Heuristic Function N1:  Manhattan Distance 

In [13]:
def manhattan_distance(state, goal):
    """
    Compute Manhattan distance between current state and goal state.
    state and goal are 3x3 lists.
    """

    # first: build lookup table for goal positions
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            val = goal[r][c]
            goal_pos[val] = (r, c)

    total = 0

    # compute sum of Manhattan distances for tiles 1..8
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val == 0:
                continue  # skip blank
            gr, gc = goal_pos[val]
            total += abs(r - gr) + abs(c - gc)

    return total

In [14]:
state = [[1, 2, 3],
         [4, 0, 5],
         [6, 7, 8]]

goal = [[2,3,5],
        [1,0,7],
        [4,6,8]]


manhattan_distance(state, goal)

8

### Heuristic Function N2:  Manhattan + 2 * linear conflict

In [15]:
def manhattan_conflict(state, goal):
    """
    Heuristic = Manhattan(state, goal) + 2 * (number of linear conflicts)

    """

    # --- Get Manhattan distance from the separate function ---
    manhattan = manhattan_distance(state, goal)

    # --- Build goal lookup table (needed for conflict checks) ---
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    # --- Count linear conflicts ---
    conflicts = 0

    # Row conflicts
    for row in range(3):
        row_tiles = []
        for col in range(3):
            tile = state[row][col]
            if tile != 0:
                gr, gc = goal_pos[tile]
                if gr == row:  # tile belongs in this row
                    row_tiles.append((col, gc))

        # count reversed pairs
        for i in range(len(row_tiles)):
            for j in range(i + 1, len(row_tiles)):
                col1, gcol1 = row_tiles[i]
                col2, gcol2 = row_tiles[j]
                if gcol1 > gcol2:
                    conflicts += 1

    # Column conflicts
    for col in range(3):
        col_tiles = []
        for row in range(3):
            tile = state[row][col]
            if tile != 0:
                gr, gc = goal_pos[tile]
                if gc == col:  # tile belongs in this column
                    col_tiles.append((row, gr))

        for i in range(len(col_tiles)):
            for j in range(i + 1, len(col_tiles)):
                row1, grow1 = col_tiles[i]
                row2, grow2 = col_tiles[j]
                if grow1 > grow2:
                    conflicts += 1

    # Final heuristic
    return manhattan + 2 * conflicts


In [16]:
state = [[1, 2, 3],
         [4, 0, 5],
         [6, 7, 8]]

goal = [[2,1,5],
        [4,7,3],
        [0,6,8]]

manhattan_conflict(state, goal)

10

In [17]:
manhattan_distance(state, goal)

6

## 3:  A Function For Printing States

In [18]:
def print_state(state):
    """Print a 3×3 puzzle matrix."""
    for row in state:
        print(" ".join(str(x) for x in row))
    print()

In [19]:
print_state(state)

1 2 3
4 0 5
6 7 8



## 4: A Function For Turning State To Tuple 

In [20]:
def state_to_tuple(state):
    """Convert list-of-lists state into an immutable tuple form (for hashing)."""
    return tuple(tuple(row) for row in state)

## 5: A Function For Reconstructing Path 

In [21]:
def reconstruct_path(came_from, current):
    """
    Reconstruct the full path by walking backward from goal to start
    using the 'came_from' dictionary.
    """
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path


## 6: A Function For Detecting Move And Swap

In [22]:
def detect_move_and_swap(prev_state, next_state):
    """
    Detect the direction of movement (up/down/left/right)
    and describe which tile was swapped with the blank (0).
    """
    p0 = n0 = None

    # Find zero positions
    for i in range(3):
        for j in range(3):
            if prev_state[i][j] == 0:
                p0 = (i, j)
            if next_state[i][j] == 0:
                n0 = (i, j)

    pi, pj = p0
    ni, nj = n0

    # Determine direction
    if ni == pi - 1 and nj == pj:
        direction = "up"
    elif ni == pi + 1 and nj == pj:
        direction = "down"
    elif ni == pi and nj == pj - 1:
        direction = "left"
    elif ni == pi and nj == pj + 1:
        direction = "right"
    else:
        direction = "unknown"

    swapped_value = prev_state[ni][nj]
    swap_info = f"swap zero with tile {swapped_value} at position ({ni},{nj})"

    return direction, swap_info



# Define A* Function

In [23]:
def a_star(start, goal, heuristic_func):
 
    start_t = state_to_tuple(start)
    goal_t = state_to_tuple(goal)

    open_heap = []
    g_score = {start_t: 0}

    f_start = heuristic_func(start, goal)
    heapq.heappush(open_heap, (f_start, 0, start_t))

    came_from = {}
    closed = set()

    # Counters
    generated_nodes = 0
    expanded_nodes = 0
    max_open_size = 1
    max_closed_size = 0

    print(f"\n=== A* SEARCH STARTED WITH  HEURISTIUC {heuristic_func.__name__} ===\n")

    while open_heap:

        max_open_size = max(max_open_size, len(open_heap))
        max_closed_size = max(max_closed_size, len(closed))

        f, g, current_t = heapq.heappop(open_heap)

        if current_t in closed:
            print("Skipped stale entry (already in CLOSED).")
            print("--------------------------------------------")
            continue

        expanded_nodes += 1
        current_state = [list(row) for row in current_t]

        print("Expanding node with f =", f, "g =", g)
        print_state(current_state)
        print("CLOSED size:", len(closed))
        print("OPEN size:", len(open_heap))
        print("-------------------------------------------------------")

        closed.add(current_t)

        # Check goal
        if current_t == goal_t:
            final_path = reconstruct_path(came_from, current_t)

            move_sequence = []
            swap_details = []

            for i in range(len(final_path) - 1):
                prev_s = [list(r) for r in final_path[i]]
                next_s = [list(r) for r in final_path[i + 1]]
                direction, swap_info = detect_move_and_swap(prev_s, next_s)

                move_sequence.append(direction)
                swap_details.append(swap_info)

            swaps = len(final_path) - 1
            space_complexity = max_open_size + max_closed_size

            # Build final report to return in next cell
            report = []
            report.append(f"===== FINAL RESULT WITH HEURISTIC {heuristic_func.__name__}  =====")
            report.append("Status: Path found successfully.")
            report.append(f"Total swaps: {swaps}")
            report.append(f"Generated nodes: {generated_nodes}")
            report.append(f"Expanded nodes: {expanded_nodes}")
            report.append(f"Space complexity: {space_complexity}")
            report.append("")
            report.append("Move sequence:")
            report.append(str(move_sequence))
            report.append("")
            report.append("Swap details:")
            for s in swap_details:
                report.append(s)
            report.append("========================")

            # Return final summary string
            return "\n".join(report)

        # Generate children
        children = generate_children(current_state)

        print("Generated", len(children), "children:")
        print("-------------------------------------------------------")

        for child in children:

            generated_nodes += 1
            child_t = state_to_tuple(child)

            if child_t in closed:
                print("Child ignored (already in CLOSED):")
                print_state(child)
                print("-------------------------------------------------------")
                continue

            tentative_g = g + 1
            child_h = heuristic_func(child, goal)
            child_f = tentative_g + child_h

            better = (
                child_t not in g_score or tentative_g < g_score[child_t]
            )

            print("Child:")
            print_state(child)
            print("g =", tentative_g, "h =", child_h, "f =", child_f)

            if not better:
                print("NOT added to OPEN (worse path).")
                print("-------------------------------------------------------")
                continue

            g_score[child_t] = tentative_g
            came_from[child_t] = current_t
            heapq.heappush(open_heap, (child_f, tentative_g, child_t))

            print("Added to OPEN.")
            print("-------------------------------------------------------")

        print("\n### END OF EXPANSION ###\n")

    # No solution found
    report = []
    report.append("===== FINAL RESULT =====")
    report.append("Status: No path exists from start to goal.")
    report.append(f"Generated nodes: {generated_nodes}")
    report.append(f"Expanded nodes: {expanded_nodes}")
    report.append(f"Space complexity: {max_open_size + max_closed_size}")
    report.append("========================")

    return "\n".join(report)


# Report & Call

## Running A* with Manhattan heuristic 

In [24]:
result = a_star(start = [[7,2,8],
                    [4,5,6],
                    [3,0,1]]
, goal=[[1,2,3],
       [7,6,5],
       [8,0,4]]
, heuristic_func= manhattan_distance)



=== A* SEARCH STARTED WITH  HEURISTIUC manhattan_distance ===

Expanding node with f = 18 g = 0
7 2 8
4 5 6
3 0 1

CLOSED size: 0
OPEN size: 0
-------------------------------------------------------
Generated 3 children:
-------------------------------------------------------
Child:
7 2 8
4 0 6
3 5 1

g = 1 h = 19 f = 20
Added to OPEN.
-------------------------------------------------------
Child:
7 2 8
4 5 6
0 3 1

g = 1 h = 17 f = 18
Added to OPEN.
-------------------------------------------------------
Child:
7 2 8
4 5 6
3 1 0

g = 1 h = 17 f = 18
Added to OPEN.
-------------------------------------------------------

### END OF EXPANSION ###

Expanding node with f = 18 g = 1
7 2 8
4 5 6
0 3 1

CLOSED size: 1
OPEN size: 2
-------------------------------------------------------
Generated 2 children:
-------------------------------------------------------
Child:
7 2 8
0 5 6
4 3 1

g = 2 h = 16 f = 18
Added to OPEN.
-------------------------------------------------------
Child ignored

In [25]:
print(result)

===== FINAL RESULT WITH HEURISTIC manhattan_distance  =====
Status: Path found successfully.
Total swaps: 26
Generated nodes: 7025
Expanded nodes: 2628
Space complexity: 4048

Move sequence:
['left', 'up', 'up', 'right', 'down', 'down', 'right', 'up', 'up', 'left', 'down', 'down', 'left', 'up', 'right', 'right', 'down', 'left', 'up', 'right', 'up', 'left', 'left', 'down', 'down', 'right']

Swap details:
swap zero with tile 3 at position (2,0)
swap zero with tile 4 at position (1,0)
swap zero with tile 7 at position (0,0)
swap zero with tile 2 at position (0,1)
swap zero with tile 5 at position (1,1)
swap zero with tile 3 at position (2,1)
swap zero with tile 1 at position (2,2)
swap zero with tile 6 at position (1,2)
swap zero with tile 8 at position (0,2)
swap zero with tile 5 at position (0,1)
swap zero with tile 3 at position (1,1)
swap zero with tile 1 at position (2,1)
swap zero with tile 4 at position (2,0)
swap zero with tile 7 at position (1,0)
swap zero with tile 1 at position

### Save a .txt File

In [26]:
with open("data/a_star_manhattan.txt", "w", encoding="utf-8") as f:
    f.write(result)

## Running A* with Manhattan_conflict heuristic 

In [27]:
result_2 = a_star(start = [[7,2,8],
                    [4,5,6],
                    [3,0,1]]
, goal=[[1,2,3],
       [7,6,5],
       [8,0,4]]
, heuristic_func= manhattan_conflict)


=== A* SEARCH STARTED WITH  HEURISTIUC manhattan_conflict ===

Expanding node with f = 20 g = 0
7 2 8
4 5 6
3 0 1

CLOSED size: 0
OPEN size: 0
-------------------------------------------------------
Generated 3 children:
-------------------------------------------------------
Child:
7 2 8
4 0 6
3 5 1

g = 1 h = 19 f = 20
Added to OPEN.
-------------------------------------------------------
Child:
7 2 8
4 5 6
0 3 1

g = 1 h = 19 f = 20
Added to OPEN.
-------------------------------------------------------
Child:
7 2 8
4 5 6
3 1 0

g = 1 h = 19 f = 20
Added to OPEN.
-------------------------------------------------------

### END OF EXPANSION ###

Expanding node with f = 20 g = 1
7 2 8
4 0 6
3 5 1

CLOSED size: 1
OPEN size: 2
-------------------------------------------------------
Generated 4 children:
-------------------------------------------------------
Child:
7 0 8
4 2 6
3 5 1

g = 2 h = 20 f = 22
Added to OPEN.
-------------------------------------------------------
Child ignored

In [28]:
print(result_2)

===== FINAL RESULT WITH HEURISTIC manhattan_conflict  =====
Status: Path found successfully.
Total swaps: 26
Generated nodes: 3768
Expanded nodes: 1440
Space complexity: 2192

Move sequence:
['left', 'up', 'up', 'right', 'down', 'down', 'right', 'up', 'up', 'left', 'down', 'down', 'left', 'up', 'right', 'right', 'down', 'left', 'up', 'right', 'up', 'left', 'left', 'down', 'down', 'right']

Swap details:
swap zero with tile 3 at position (2,0)
swap zero with tile 4 at position (1,0)
swap zero with tile 7 at position (0,0)
swap zero with tile 2 at position (0,1)
swap zero with tile 5 at position (1,1)
swap zero with tile 3 at position (2,1)
swap zero with tile 1 at position (2,2)
swap zero with tile 6 at position (1,2)
swap zero with tile 8 at position (0,2)
swap zero with tile 5 at position (0,1)
swap zero with tile 3 at position (1,1)
swap zero with tile 1 at position (2,1)
swap zero with tile 4 at position (2,0)
swap zero with tile 7 at position (1,0)
swap zero with tile 1 at position

### Save a .txt File

In [29]:
with open("data/a_star_manhattan_conflict.txt", "w", encoding="utf-8") as f:
    f.write(result_2)